# NB18 s0 — EEGNet with cross-subject dropout 0.25 (GPU, hard cap 0.75 h)

Same EEGNet-8,2, splits, training rule and aggregation as NB06, with dropout 0.25 (the original paper's cross-subject setting) instead of the shared 0.5; compared with the NB06 EEGNet and the GCN in its own Holm family, and added to the equivalence table as a separate family.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_could.py tests/test_equivalence.py tests/test_model_train.py', log=f'{WORK}/pytest_nb18.txt')

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)
import torch
assert torch.cuda.is_available(), 'no GPU attached: refusing to run on CPU'
print(torch.cuda.get_device_name(0))

In [ ]:
sh(f'{PY} grid --cache {CACHE} --folds {FOLDS} --pipelines P5 --edges identity --tag deep_eegnet_drop025 --set model.arch=eegnet model.dropout=0.25 --device cuda --out {WORK}/deep_sensitivity/eegnet_drop025_runs.jsonl --max-hours 0.75', log=f'{WORK}/eegnet_drop025.log')

In [ ]:
sh(f'{PY} deep-dropout --runs {WORK}/deep_sensitivity/eegnet_drop025_runs.jsonl --deep-runs {SRC}/reports/deep/deep_runs_s0.jsonl --main-runs {SRC}/reports/grid/runs_s0.jsonl {SRC}/reports/grid/runs_s1.jsonl {SRC}/reports/grid/runs_s2.jsonl --out {WORK}/deep_sensitivity', log=f'{WORK}/deep_dropout.log')

In [ ]:
import shutil
shutil.copytree(f'{SRC}/reports', f'{WORK}/reports_tmp')
shutil.copytree(f'{WORK}/deep_sensitivity', f'{WORK}/reports_tmp/deep_sensitivity')
sh(f'{PY} equivalence --reports {WORK}/reports_tmp --out {WORK}/equivalence', log=f'{WORK}/equivalence.log')
shutil.rmtree(f'{WORK}/reports_tmp')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)